# Notebook 04 — Statistical Analysis

> Chi-square, T-test, Pearson Correlation, ANOVA, Mann-Whitney, Linear Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

matches = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')
print('Loaded.')

## Test 1 — Chi-Square: Toss Decision vs Match Outcome

**H0**: Toss decision has no effect on match outcome
**H1**: Toss decision significantly affects match outcome

In [ ]:
valid = matches[matches['winner'].notna()].copy()
ct = pd.crosstab(valid['toss_decision'], valid['toss_win_match'])
chi2, p_val, dof, expected = stats.chi2_contingency(ct)
print(f'Chi2={chi2:.4f}, p={p_val:.4f}, dof={dof}')
print(f'Decision: {"REJECT H0" if p_val < 0.05 else "FAIL TO REJECT H0"} at 5% level')
print(ct)

## Test 2 — T-Test: Powerplay Runs — Match Winners vs Losers

In [ ]:
pp = deliveries[deliveries['phase']=='Powerplay']
pp_scores = pp.groupby('match_id')['total_runs'].sum().reset_index()
pp_scores.columns = ['match_id','pp_runs']
ing1_bat = deliveries[deliveries['inning']==1].groupby('match_id')['batting_team'].first().reset_index()
ing1_bat.columns = ['match_id','bat_team']
pp_scores = pp_scores.merge(matches[['id','winner']], left_on='match_id', right_on='id')
pp_scores = pp_scores.merge(ing1_bat, on='match_id')
pp_scores['bat_won'] = (pp_scores['bat_team'] == pp_scores['winner']).astype(int)
won_pp = pp_scores[pp_scores['bat_won']==1]['pp_runs']
lost_pp = pp_scores[pp_scores['bat_won']==0]['pp_runs']
t_stat, p_t = stats.ttest_ind(won_pp, lost_pp)
print(f'Winner PP avg: {won_pp.mean():.2f}, Loser PP avg: {lost_pp.mean():.2f}')
print(f't={t_stat:.4f}, p={p_t:.4f}')
print(f'Decision: {"REJECT H0" if p_t < 0.05 else "FAIL TO REJECT H0"}')

## Test 3 — Pearson Correlation: 1st Innings Score vs Win Margin

In [ ]:
first_inn = deliveries[deliveries['inning']==1].groupby('match_id')['total_runs'].sum().reset_index()
first_inn.columns = ['match_id','score_1']
merged = first_inn.merge(matches[['id','result_margin','result']], left_on='match_id', right_on='id')
merged = merged[merged['result']=='runs'].dropna()
corr, p_corr = stats.pearsonr(merged['score_1'], merged['result_margin'])
print(f'Pearson r={corr:.4f}, p={p_corr:.6f}')
# Regression
slope, intercept, r_val, p_reg, se = stats.linregress(merged['score_1'], merged['result_margin'])
print(f'Regression: slope={slope:.4f}, R2={r_val**2:.4f}')
print(f'Every 10 extra runs -> +{slope*10:.2f} run win margin')

## Test 4 — ANOVA: Scoring Across Top 8 Venues

In [ ]:
top_venues = matches['venue'].value_counts().head(8).index.tolist()
scores_venue = first_inn.merge(matches[['id','venue']], left_on='match_id', right_on='id')
scores_venue = scores_venue[scores_venue['venue'].isin(top_venues)]
groups = [grp['score_1'].values for _, grp in scores_venue.groupby('venue')]
f_stat, p_anova = stats.f_oneway(*groups)
print(f'F={f_stat:.4f}, p={p_anova:.4f}')
print(f'Decision: {"REJECT H0" if p_anova < 0.05 else "FAIL TO REJECT H0"} — venues do{" " if p_anova<0.05 else " not "}significantly differ')

## Correlation Heatmap

In [ ]:
corr_data = deliveries.groupby('match_id').agg(
    total_runs=('total_runs','sum'),
    wickets=('is_wicket','sum'),
    extras=('extra_runs','sum'),
    fours=('batsman_runs', lambda x: (x==4).sum()),
    sixes=('batsman_runs', lambda x: (x==6).sum()),
).reset_index()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_data[['total_runs','wickets','extras','fours','sixes']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Matrix — Match-Level Variables', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figures/fig11_correlation_heatmap.png', dpi=150); plt.show()